# Doublet Detection for snRNA-seq Datasets

Scrublet-based doublet detection with per-cluster score aggregation
(overclustering + median cluster score + BH/Bonferroni p-values).

In [ ]:
import os

import numpy as np
import scanpy as sc
import scipy.stats
import scrublet as scr

sc.settings.verbosity = 0  # errors (0), warnings (1), info (2), hints (3)
sc.settings.set_figure_params(dpi=80)

## FDR correction helpers

In [ ]:
def bh(pvalues):
    """Benjamini-Hochberg FDR correction."""
    pvalues = np.array(pvalues)
    n = int(pvalues.shape[0])
    new_pvalues = np.empty(n)
    values = [(pvalue, i) for i, pvalue in enumerate(pvalues)]
    values.sort()
    values.reverse()
    new_values = []
    for i, vals in enumerate(values):
        rank = n - i
        pvalue, index = vals
        new_values.append((n / rank) * pvalue)
    for i in range(0, int(n) - 1):
        if new_values[i] < new_values[i + 1]:
            new_values[i + 1] = new_values[i]
    for i, vals in enumerate(values):
        pvalue, index = vals
        new_pvalues[index] = new_values[i]
    return new_pvalues


def bonf(pvalues):
    """Bonferroni FDR correction."""
    new_pvalues = np.array(pvalues) * len(pvalues)
    new_pvalues[new_pvalues > 1] = 1
    return new_pvalues

## Config

Set sample IDs and input/output directories.

In [ ]:
# List of sample IDs to process (each expected at:
# {data_dir}/{sample_id}/{sample_id}_sn10x3GEX/outs/cellbender_output_filtered.h5)
sample_ids = ["SAMPLE1", "SAMPLE2"]

data_dir = "/path/to/cellranger/fastqs/"
output_dir = "/path/to/results/scrublet-scores/"

scorenames = ["scrublet_score", "scrublet_cluster_score", "zscore", "bh_pval", "bonf_pval"]

## Pipeline functions

In [ ]:
def load_and_qc(h5_path, sample_id, min_genes=200, min_cells=3, max_mito=0.05):
    """Load a 10x h5 matrix and apply light QC filtering."""
    adata = sc.read_10x_h5(h5_path)
    adata.var_names_make_unique()
    adata.obs_names = [f"{sample_id}_{bc}" for bc in adata.obs_names]

    sc.pp.filter_cells(adata, min_genes=min_genes)
    sc.pp.filter_genes(adata, min_cells=min_cells)

    mito_genes = [name for name in adata.var_names if name.lower().startswith("mt-")]
    adata.obs["percent_mito"] = (
        np.sum(adata[:, mito_genes].X, axis=1).A1 / np.sum(adata.X, axis=1).A1
    )
    adata = adata[adata.obs["percent_mito"] < max_mito, :].copy()
    return adata


def run_scrublet(adata, seed=0):
    """Run Scrublet and attach per-cell doublet scores."""
    np.random.seed(seed)
    scrub = scr.Scrublet(adata.X)
    doublet_scores, _ = scrub.scrub_doublets(verbose=False)
    adata.obs["scrublet_score"] = doublet_scores
    return adata


def overcluster(adata):
    """Basic PCA/neighbors/Leiden pipeline, then a second round of Leiden
    within each first-round cluster ("overclustering")."""
    adata.X = adata.X.astype(np.float64)
    sc.pp.normalize_per_cell(adata, counts_per_cell_after=1e4)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
    adata = adata[:, adata.var["highly_variable"]].copy()
    sc.pp.scale(adata, max_value=10)
    sc.tl.pca(adata, svd_solver="arpack")
    sc.pp.neighbors(adata)
    sc.tl.leiden(adata)
    adata.obs["leiden"] = [str(i) for i in adata.obs["leiden"]]

    for clus in np.unique(adata.obs["leiden"]):
        adata_sub = adata[adata.obs["leiden"] == clus].copy()
        sc.tl.leiden(adata_sub)
        adata_sub.obs["leiden"] = [f"{clus},{i}" for i in adata_sub.obs["leiden"]]
        adata.obs.loc[adata_sub.obs_names, "leiden"] = adata_sub.obs["leiden"]
    return adata


def score_clusters(adata):
    """Compute per-cluster median Scrublet score, then z-scores and
    BH/Bonferroni p-values across clusters."""
    for clus in np.unique(adata.obs["leiden"]):
        mask = adata.obs["leiden"] == clus
        adata.obs.loc[mask, "scrublet_cluster_score"] = np.median(
            adata.obs.loc[mask, "scrublet_score"]
        )

    med = np.median(adata.obs["scrublet_cluster_score"])
    above_med = adata.obs["scrublet_cluster_score"] > med
    mad = np.median(adata.obs["scrublet_cluster_score"][above_med] - med)

    zscores = (adata.obs["scrublet_cluster_score"].values - med) / (1.4826 * mad)
    adata.obs["zscore"] = zscores

    pvals = 1 - scipy.stats.norm.cdf(zscores)
    adata.obs["bh_pval"] = bh(pvals)
    adata.obs["bonf_pval"] = bonf(pvals)
    return adata


def process_sample(sample_id, data_dir, output_dir):
    """Run the full doublet-detection pipeline for one sample and write
    a CSV of scores to output_dir."""
    h5_path = os.path.join(
        data_dir, sample_id, f"{sample_id}_sn10x3GEX", "outs", "cellbender_output_filtered.h5"
    )
    adata = load_and_qc(h5_path, sample_id)
    adata = run_scrublet(adata)
    adata = overcluster(adata)
    adata = score_clusters(adata)

    scores = adata.obs[scorenames]
    os.makedirs(output_dir, exist_ok=True)
    scores.to_csv(os.path.join(output_dir, f"{sample_id}.csv"))
    return scores

## Run for all samples

In [ ]:
for sample_id in sample_ids:
    print(sample_id)
    process_sample(sample_id, data_dir, output_dir)